In [1]:

import asyncio
import warnings
import pandas as pd
from pathlib import Path

# Парсеры
from antimony import antimony_parser
from westmetall import westmetall_async
from lme import lme_selenium_async
from lbma import lbma_prescious_async

from cbr import cb_currency, cb_metalls
from nbk import nbk_tenge_async
from shmet import shmet_optimized_async
from kitco import kitco_parser_async

# Сервисные функции из service_layer
from service_layer import (
    read_db,
    save_db,
    check_df,
    check_and_save_pair,
    show_db,
    excel_to_csv_db
)

warnings.filterwarnings("ignore")


# ================================================================
# Пути к базам (реальная структура проекта)
# ================================================================
LME_PATH = Path("lme/data/LME_db_new.xlsx")
WESTMETALL_PATH = Path("westmetall/data/LME_westmetall_db.xlsx")

KITCO_PATH = Path("kitco/data/kitko_db.xlsx")
LBMA_PATH = Path("lbma/data/lbma_kitco_subs.xlsx")

ANTIMONY_PATH = Path("antimony/data/antimony.xlsx")

CB_CURRENCY_PATH = Path("cbr/data/cb_currency.xlsx")
CB_METALLS_PATH = Path("cbr/data/cb_metalls.xlsx")

NBK_PATH = Path("nbk/data/nbk_tenge.xlsx")
SHMET_PATH = Path("shmet/data/shmet_historical.xlsx")


# ================================================================
# Проверка целостности парных баз
# ================================================================
def db_check():
    """
    Проверка целостности парных баз с очисткой дубликатов
    и сохранением первого (более раннего) вхождения.
    """
    print("Проверка LME / Westmetall...")
    check_and_save_pair(
        LME_PATH,
        WESTMETALL_PATH,
        pair_name="LME / Westmetall",
        index=False,
    )

    print("Проверка Kitco / LBMA...")
    check_and_save_pair(
        KITCO_PATH,
        LBMA_PATH,
        pair_name="Kitco / LBMA",
        index=False,
    )
    
async def main():
    print("Parsing started...")

    tasks = {
        "lme": lme_selenium_async(),
        "antimony": antimony_parser(),
        "westmetall": westmetall_async(),
        "lbma": lbma_prescious_async(),
        "kitco": kitco_parser_async(),
        "cb_currency": cb_currency(),
        "cb_metalls": cb_metalls(),
        "nbk": nbk_tenge_async(),
        "shmet": shmet_optimized_async(),
    }

    # return_exceptions=True — чтобы падение одного парсера
    # не останавливало остальные
    results = await asyncio.gather(
        *tasks.values(),
        return_exceptions=True,
    )

    for name, result in zip(tasks.keys(), results):
        if isinstance(result, Exception):
            print(f"❌ Ошибка в {name}: {result}")

    print("All tasks are done!")
    print("+" * 64)
    print("Checking DB...")

    db_check()

    print("DB check completed!")
    print("+" * 64)
    print("Visual control")
    print("+" * 64)
    
    print("Converting Excel to CSV...")
    excel_to_csv_db()
    print("CSV conversion completed!")

    # Базовые металлы
    show_db("lme_selenium_db", LME_PATH, sheet_name=0)
    show_db("westmetall_db", WESTMETALL_PATH, sheet_name=0)

    # Драгоценные металлы
    show_db("kitco_db", KITCO_PATH, sheet_name=0)
    show_db("lbma_precious_db", LBMA_PATH, sheet_name=0)

    # Антимоний
    show_db("antimony_db", ANTIMONY_PATH, sheet_name=0)

    # ЦБ РФ: валюты (каждая на своем листе)
    for currency in [
        "USD",
        "EUR",
        "British_Pound",
        "China_Yuan",
        "Japanese_Yen",
        "Swiss_Franc",
    ]:
        show_db(
            f"cb_currency ({currency})",
            CB_CURRENCY_PATH,
            sheet_name=currency,
        )

    # ЦБ РФ: металлы
    show_db("cb_metalls_db", CB_METALLS_PATH, sheet_name=0)

    # Казахстан и SHMET
    show_db("nbk_tenge_db", NBK_PATH, sheet_name=0)
    show_db("shmet_historical_db", SHMET_PATH, sheet_name=0, show_head=True)


# Для Jupyter используем await, а не asyncio.run()
await main()

Parsing started...
🚀 LME parsing started...
antimony parsing is DONE
NBK_tenge parsing is DONE! (1747 строк)
✅ LME_main is done!!!
CB_metalls parsing is DONE!
USD is done!
EUR is done!
Australian_Dollar is done!
China_Yuan is done!
British_Pound is done!
Kazakhstan_Tenge is done!
Japanese_Yen is done!
Swiss_Franc is done!
CB_currency parsing is DONE!
Произошла ошибка KITCO: Не найдены блоки <div class='grid'> на странице Kitco
WESTMETALL is done!!!
SHMET is done!!!
[LBMA/Selenium] Открываем страницу LBMA...
[LBMA/Selenium] Обрабатываем Gold...
[LBMA/Selenium] Сетевых ответов от prices.lbma.org.uk: 1
[LBMA/Selenium] Gold: получено 10 строк
[LBMA/Selenium] Обрабатываем Silver...
[LBMA/Selenium] Сетевых ответов от prices.lbma.org.uk: 1
[LBMA/Selenium] Silver: получено 10 строк
[LBMA/Selenium] Обрабатываем Platinum...
[LBMA/Selenium] Сетевых ответов от prices.lbma.org.uk: 2
[LBMA/Selenium] Platinum: получено 10 строк
[LBMA/Selenium] Обрабатываем Palladium...
[LBMA/Selenium] Сетевых ответов

,date,lead,nickel,zink,tin,aluminium,copper
1226,2026-11-09,1854.0,16190,3754.0,46350,3147.0,14030.0
1227,2026-11-10,1846.0,15875,3768.0,46345,3137.0,13992.0
1228,2026-11-11,1845.0,15930,3794.0,47200,3223.5,13875.5
1229,2026-11-12,1843.0,16010,3760.5,46650,3155.5,13712.0
1230,2026-11-13,1853.0,16025,3745.0,48295,3150.5,13610.0


westmetall_db


,date,aluminium,copper,lead,nickel,zink,tin
1226,2026-11-09,3147.0,14030.0,1854.0,16190,3754.0,46350
1227,2026-11-10,3137.0,13992.0,1846.0,15875,3768.0,46345
1228,2026-11-11,3223.5,13875.5,1845.0,15930,3794.0,47200
1229,2026-11-12,3155.5,13712.0,1843.0,16010,3760.5,46650
1230,2026-11-13,3150.5,13610.0,1853.0,16025,3745.0,48295


kitco_db


,Date,Gold,Silver,Platinum,Palladium
14887,2026-11-09,4117.00,52.625,1560.60,1028.00
14888,2026-11-10,4050.90,52.285,1553.10,1037.25
14889,2026-11-11,4014.00,52.730,1563.65,1078.75
14890,2026-11-12,3870.20,50.925,1522.00,1011.10
14891,2026-11-13,3890.85,50.045,1507.65,1013.70


... (18 outputs left out)

nbk_tenge_db


,date,Числовое значение,ДОЛЛАР США
1742,2026-11-10,1,448.98
1743,2026-11-11,1,447.52
1744,2026-11-12,1,444.76
1745,2026-11-13,1,443.62
1746,2026-11-14,1,444.30


shmet_historical_db


,date,price,unit
0,2020-01-10,48605,Yuan/MT
1,2020-01-14,48990,Yuan/MT
2,2020-01-15,49060,Yuan/MT
3,2020-01-16,48950,Yuan/MT
4,2020-01-17,48930,Yuan/MT
